In [9]:
import pandas as pd
import re

df = pd.read_csv('data_labeled.csv')   # dari Tugas 2

print('Dimensi data:', df.shape)
print('\n5 baris pertama:')
print(df.head())
print('\nDistribusi sentimen:')
print(df['sentimen'].value_counts())

Dimensi data: (1064, 6)

5 baris pertama:
                                     id  \
0  d358287c-0a81-49a4-9a13-7e309b9ca0f2   
1  e089a938-de66-4061-a2af-6090b25f9e8c   
2  fbfbe930-b968-438d-93cf-c7224ffa226e   
3  2aa126bf-d5bb-4476-a37f-0494db40b07e   
4  20c61ebb-c0db-4201-bbf8-a5c04cc906b0   

                                                teks              tanggal  \
0                    gabisa login diwajibkan premium  2026-08-20 06:28:24   
1                                              jelek  2026-08-20 06:28:01   
2                            ngapa berbayar si? aneh  2026-08-20 06:14:41   
3  tidak suka versi yang sekarang. untuk membuat ...  2026-08-20 06:09:35   
4  semua bayar mana banyak errornya lgi pantesan ...  2026-08-20 06:01:25   

      sumber sentimen_auto sentimen  
0  playstore        netral  negatif  
1  playstore       negatif  negatif  
2  playstore        netral  negatif  
3  playstore       positif  negatif  
4  playstore       negatif  negatif  

Distrib

In [10]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()                                  # case folding
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # hapus URL
    text = re.sub(r'@\w+|#\w+', '', text)               # hapus mention/hashtag
    text = re.sub(r'\d+', '', text)                      # hapus angka
    text = re.sub(r'[^\w\s]', '', text)                 # hapus tanda baca
    text = re.sub(r'\s+', ' ', text).strip()             # spasi berlebih
    return text

df['teks_clean'] = df['teks'].apply(clean_text)
print(df[['teks', 'teks_clean']].head())

                                                teks  \
0                    gabisa login diwajibkan premium   
1                                              jelek   
2                            ngapa berbayar si? aneh   
3  tidak suka versi yang sekarang. untuk membuat ...   
4  semua bayar mana banyak errornya lgi pantesan ...   

                                          teks_clean  
0                    gabisa login diwajibkan premium  
1                                              jelek  
2                             ngapa berbayar si aneh  
3  tidak suka versi yang sekarang untuk membuat c...  
4  semua bayar mana banyak errornya lgi pantesan ...  


In [11]:
informal_words_dict = {
    'gabisa': 'tidak bisa',
    'lgi': 'lagi',
    'gk': 'tidak',
    'yg': 'yang',
    'udh': 'sudah',
    'bgt': 'banget',
    'tdk': 'tidak',
    'ga': 'tidak',
    'aja': 'saja',
    'bikin': 'membuat',
    'bgs': 'bagus',
    'udh': 'sudah',
    'ngapa': 'mengapa',
    'hrs': 'harus',
    'skrg': 'sekarang',
    'apk': 'aplikasi',
    'tele': 'telegram',
    'utk': 'untuk',
    'k': 'ke',
    'ttep': 'tetep',
    'g bs': 'tidak bisa',
    'dalem': 'dalam',
    'app': 'aplikasi',
    'tp': 'tapi',
    'pass': 'password',
    'mksd': 'maksud',
    'tbtb': 'tiba-tiba',
    'gimna': 'gimana',
    'ni': 'ini',
    'dev': 'developer',
    'pda':'pada',
    'knp':'kenapa',
    'br': 'baru',
    'rb': 'ribu',
    'prem': 'premium',
    'mnta': 'minta',
    'byar': 'bayar',
    'sich': 'sih',
    'ngk': 'tidak',
    'apl': 'aplikasi',
    'nggk': 'tidak',
    'aplg': 'apalagi',
    'taun': 'tahun',
    'dlm': 'dalam',
    'aj':'aja',
    'binta':'bintang',
    'bnr':'benar',
    'sya':'saya',
    'blm':'belum'
}

def normalize_informal_words(text):
    words = text.split()
    normalized_words = [informal_words_dict.get(word, word) for word in words]
    return ' '.join(normalized_words)

# Terapkan normalisasi ke kolom teks_clean
df['teks_normalized'] = df['teks_clean'].apply(normalize_informal_words)

print(df[['teks_clean', 'teks_normalized']].head())

                                          teks_clean  \
0                    gabisa login diwajibkan premium   
1                                              jelek   
2                             ngapa berbayar si aneh   
3  tidak suka versi yang sekarang untuk membuat c...   
4  semua bayar mana banyak errornya lgi pantesan ...   

                                     teks_normalized  
0                tidak bisa login diwajibkan premium  
1                                              jelek  
2                           mengapa berbayar si aneh  
3  tidak suka versi yang sekarang untuk membuat c...  
4  semua bayar mana banyak errornya lagi pantesan...  


In [12]:
pip install Sastrawi

In [13]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from nltk.corpus import stopwords

stop_words_indonesia = set(stopwords.words('indonesian'))

def remove_stopwords_from_text(text, stopwords_set):
    if not isinstance(text, str):
        return ""
    words = text.split()
    filtered_words = [word for word in words if word not in stopwords_set]
    return ' '.join(filtered_words)

In [8]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Initialize the stemmer
stemmer = StemmerFactory().create_stemmer()

# Update the preprocess_full function to use NLTK stopwords

def preprocess_full_with_nltk(text):
    text = clean_text(text)              # cleaning
    text = normalize_informal_words(text) # normalisasi bahasa kasual
    text = remove_stopwords_from_text(text, stop_words_indonesia) # hapus stop word dengan NLTK
    text = stemmer.stem(text)            # stemming
    return text

df['teks_processed_nltk'] = df['teks'].apply(preprocess_full_with_nltk)
print(df[['teks', 'teks_processed_nltk']].head())

# Simpan hasil
df.to_csv('data_preprocessed_nltk.csv', index=False)

                                                teks  \
0                    gabisa login diwajibkan premium   
1                                              jelek   
2                            ngapa berbayar si? aneh   
3  tidak suka versi yang sekarang. untuk membuat ...   
4  semua bayar mana banyak errornya lgi pantesan ...   

                                 teks_processed_nltk  
0                                login wajib premium  
1                                              jelek  
2                                      bayar si aneh  
3                            suka versi cerita bayar  
4  bayar errornya pantesan ceo lu kena tangkep be...  


In [15]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stemmer = StemmerFactory().create_stemmer()
stopword = StopWordRemoverFactory().create_stop_word_remover()

def preprocess_full(text):
    text = clean_text(text)              # cleaning
    text = normalize_informal_words(text) # normalisasi bahasa kasual
    text = stopword.remove(text)         # hapus stop word
    text = stemmer.stem(text)            # stemming
    return text

df['teks_processed'] = df['teks'].apply(preprocess_full)
print(df[['teks', 'teks_processed']].head())

# Simpan hasil
df.to_csv('data_preprocessed.csv', index=False)

                                                teks  \
0                    gabisa login diwajibkan premium   
1                                              jelek   
2                            ngapa berbayar si? aneh   
3  tidak suka versi yang sekarang. untuk membuat ...   
4  semua bayar mana banyak errornya lgi pantesan ...   

                                      teks_processed  
0                           bisa login wajib premium  
1                                              jelek  
2                                      bayar si aneh  
3        suka versi sekarang buat cerita harus bayar  
4  semua bayar mana banyak errornya pantesan ceo ...  
